In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [12]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

## image scale from (0,1) then normalize our data => (-1, 1) 

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    
    
])

trainset = CIFAR10(root="./", train=True, transform=transform, download=True)
testset = CIFAR10(root="./", train=False, transform=transform, download=True)

In [13]:
train_loader = DataLoader(trainset, batch_size=64, shuffle=True)
test_loader = DataLoader(testset, batch_size=64, shuffle=True)

## build the CNN

In [20]:
class CNN(nn.Module):
    def __init__(self):
        super (CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel_size = 2, stride = 2  

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),


            nn.Linear(256, 10)
        )


    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [29]:
model=CNN()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Train the CNN

In [43]:
epochs = 10
val_losses = []

for epoch in range (epochs):
    epoch_training_loss = 0.0
    model.train()

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criteria(output, labels)  ## loss fnx
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

        ## validation set

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            output = model.forward(images)
            loss = criteria(output, labels)
            running_val_loss += loss.item()
                
                
                
    epoch_val_loss = running_val_loss/len(test_loader)
        
    val_losses.append(epoch_val_loss)
        
        
        

    

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(train_loader)} & val_losses= {epoch_val_loss}")

    

epoch=1/10 & loss=0.0945719055345525 & val_losses= 1.3646177311611782
epoch=2/10 & loss=0.09506167335581044 & val_losses= 1.3738270994204624
epoch=3/10 & loss=0.08724333516731762 & val_losses= 1.4245730452476792
epoch=4/10 & loss=0.07642768132631354 & val_losses= 1.5349372486957604
epoch=5/10 & loss=0.06603150944942442 & val_losses= 1.6481173289049962
epoch=6/10 & loss=0.07021957398880549 & val_losses= 1.7029217557542642
epoch=7/10 & loss=0.07279895472279548 & val_losses= 1.7409435130987958
epoch=8/10 & loss=0.0684154649356516 & val_losses= 1.7013057648755943
epoch=9/10 & loss=0.05721700041542363 & val_losses= 1.8097678411538434
epoch=10/10 & loss=0.061638154789059284 & val_losses= 1.908194267066421


In [35]:
correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        output = model.forward(images)
        _,predicted = torch.max(output, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print("accuracy: ",correct_labels/total_labels * 100) 

accuracy:  75.21
